# [Open in Colab](https://colab.research.google.com/github/dmochow/BME-i9400-2026/blob/main/notebooks/meeting09_evaluating_classifiers.ipynb)

# BME i9400 — Meeting 9
## Linear Classification II: Evaluating a Classifier

**Fall 2026 · Tuesday, October 13**

Today's lecture is about evaluating a classifier: the confusion matrix, the quantities derived from
it, the threshold that produces it, and the two curves — ROC and precision–recall — that describe a
classifier across all thresholds at once. The lecture closes with class imbalance, which affects
every one of these.
**Agenda**

1. The model under evaluation
2. The confusion matrix
3. Metrics from the four cells
4. The threshold is a choice
5. The ROC curve and the AUC
6. Precision, recall, and prevalence
7. Class imbalance
8. Connection to the course

---
## 1 — The model under evaluation

The cell below rebuilds the six-feature logistic regression of Meeting 8 — the same cleaning, the
same stratified split, the same standardisation — and keeps two arrays for the rest of the lecture:

- `y_test`, the true labels of the 145 held-out women, 50 of whom developed diabetes
- `p_test`, the model's fitted probability of diabetes for each of them

Everything today is computed from these two arrays. The model is fixed: evaluation changes no
weights. Nothing below uses the training set, because every number is meant to estimate what the
model does on women it has not seen.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
plt.rcParams.update({"font.size": 12, "figure.dpi": 110})

URL = "https://raw.githubusercontent.com/dmochow/BME-i9400-2026/main/data/diabetes.csv"
raw = pd.read_csv(URL)
df  = raw[(raw["Glucose"] > 0) & (raw["BloodPressure"] > 0) & (raw["BMI"] > 0)].reset_index(drop=True)
features = ["Pregnancies", "Glucose", "BloodPressure", "BMI", "DiabetesPedigreeFunction", "Age"]
X, y = df[features].values, df["Outcome"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=9400, stratify=y)
mu, sd = X_train.mean(axis=0), X_train.std(axis=0, ddof=1)
Z_train, Z_test = (X_train - mu) / sd, (X_test - mu) / sd

model  = LogisticRegression(penalty=None, max_iter=1000).fit(Z_train, y_train)
p_test = model.predict_proba(Z_test)[:, 1]

print(f"test set: {len(y_test)} women, {y_test.sum()} with diabetes (prevalence {100*y_test.mean():.1f}%)")
print(f"fitted probabilities: smallest {p_test.min():.3f}, median {np.median(p_test):.3f}, largest {p_test.max():.3f}")
print("\nfirst eight women:   probability   label")
for p, t in list(zip(p_test, y_test))[:8]:
    print(f"{'':21}{p:11.3f}{t:8d}")

---
## 2 — The confusion matrix

A decision is made for each woman by comparing her probability with a threshold, $t = 1/2$ to begin
with. Each woman then falls into one of the four cells of the truth table of Meeting 2, with the
classifier's decision in place of the test result:

| | predicted $1$ | predicted $0$ |
|---|---|---|
| **actual $1$** | true positive (TP) | false negative (FN) |
| **actual $0$** | false positive (FP) | true negative (TN) |

> This table is the **confusion matrix**. Conventions differ between sources on which axis holds the
> truth and in which order the classes appear. Scikit-learn's `confusion_matrix` puts the true class
> in the rows and the predicted class in the columns, both in the order $0, 1$, so its top-left
> entry is TN. The labels must be read before the numbers.

The two kinds of error are the two of Meeting 2. A false negative is a woman who will develop
diabetes and was told she would not; a false positive is a woman who will not and was told she
might. The matrix keeps them apart; accuracy adds them together.

In [ ]:
from sklearn.metrics import confusion_matrix

t    = 0.5
pred = (p_test >= t).astype(int)

TP = np.sum((pred == 1) & (y_test == 1))
FN = np.sum((pred == 0) & (y_test == 1))
FP = np.sum((pred == 1) & (y_test == 0))
TN = np.sum((pred == 0) & (y_test == 0))

print(f"threshold {t}\n")
print(f"{'':<12}{'predicted 1':>14}{'predicted 0':>14}")
print(f"{'actual 1':<12}{TP:>14}{FN:>14}")
print(f"{'actual 0':<12}{FP:>14}{TN:>14}")
print(f"\nscikit-learn's layout (rows: actual 0, 1; columns: predicted 0, 1):\n{confusion_matrix(y_test, pred)}")

---
## 3 — Metrics from the four cells

Every standard summary of a classifier's decisions is a ratio of these four counts. The definitions
are those of Meetings 2 and 3, under the names used in machine learning.

| Quantity | Other names | Formula | Reads along | Depends on prevalence |
|---|---|---|---|---|
| **Sensitivity** | recall, true positive rate (TPR) | $TP / (TP + FN)$ | the actual-positive row | no |
| **Specificity** | true negative rate (TNR) | $TN / (TN + FP)$ | the actual-negative row | no |
| **False positive rate** | FPR, $1 -$ specificity | $FP / (FP + TN)$ | the actual-negative row | no |
| **Precision** | positive predictive value (PPV) | $TP / (TP + FP)$ | the predicted-positive column | yes |
| **Negative predictive value** | NPV | $TN / (TN + FN)$ | the predicted-negative column | yes |
| **Accuracy** | | $(TP + TN) / n$ | the whole table | yes |
| **Balanced accuracy** | | $\tfrac{1}{2}(\text{sensitivity} + \text{specificity})$ | both rows | no |
| **F1 score** | | $2\,TP / (2\,TP + FP + FN)$ | | yes |

> **Precision** and **recall** are the names used in information retrieval and machine learning for
> PPV and sensitivity. The pair "sensitivity and specificity" describes the classifier; the pair
> "precision and recall" describes its positive predictions. Both pairs appear in the literature and
> in scikit-learn.
>
> The **F1 score** is the harmonic mean of precision and recall. It is high only when both are high,
> and it ignores the true negatives entirely, which is why it is favoured when the positive class is
> rare.
The row quantities — sensitivity, specificity, and their combinations — are computed within one
actual class and do not change when the class sizes change. The column quantities — precision and
NPV — and accuracy mix the two classes in the denominator, and so inherit the prevalence of the
evaluation set, exactly as in Meeting 2. A precision reported without a prevalence cannot be
interpreted.

The reference point for accuracy is the classifier that uses no features at all and predicts the
majority class for everyone. When the positive class is the minority its accuracy is
$1 - \text{prevalence}$: $65.5\%$ on this test set, with a sensitivity of $0$.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, balanced_accuracy_score

def summarise(y_true, p, t):
    """Every quantity of the table, for the decisions p >= t."""
    pred = (p >= t).astype(int)
    TP = np.sum((pred == 1) & (y_true == 1)); FN = np.sum((pred == 0) & (y_true == 1))
    FP = np.sum((pred == 1) & (y_true == 0)); TN = np.sum((pred == 0) & (y_true == 0))
    return {"threshold": t, "TP": TP, "FN": FN, "FP": FP, "TN": TN,
            "sensitivity": TP / (TP + FN), "specificity": TN / (TN + FP), "FPR": FP / (FP + TN),
            "precision": TP / (TP + FP) if TP + FP else np.nan, "NPV": TN / (TN + FN) if TN + FN else np.nan,
            "accuracy": (TP + TN) / len(y_true), "balanced accuracy": 0.5 * (TP / (TP + FN) + TN / (TN + FP)),
            "F1": 2 * TP / (2 * TP + FP + FN)}

s = summarise(y_test, p_test, 0.5)
for k, v in s.items():
    print(f"  {k:<18} {v:.3f}" if isinstance(v, float) else f"  {k:<18} {v}")

# the same numbers from scikit-learn
assert np.isclose(s["precision"], precision_score(y_test, pred)) and np.isclose(s["sensitivity"], recall_score(y_test, pred))
assert np.isclose(s["F1"], f1_score(y_test, pred)) and np.isclose(s["balanced accuracy"], balanced_accuracy_score(y_test, pred))

trivial = summarise(y_test, np.zeros_like(p_test), 0.5)        # never predicts the positive class
print(f"\nmajority-class classifier: accuracy {trivial['accuracy']:.3f}, sensitivity {trivial['sensitivity']:.3f}, "
      f"specificity {trivial['specificity']:.3f}")

---
## 4 — The threshold is a choice

Nothing in the model fixes the threshold at $1/2$. Lowering it calls more women positive: sensitivity
rises and specificity falls. Raising it does the opposite. Each threshold produces a different
confusion matrix, and the table of section 3 was a single row of a much longer table.

> The pair (sensitivity, specificity) that a threshold produces is called an **operating point**.
> Choosing a threshold is choosing an operating point.

Three ways to choose, in increasing order of formality:

- **A required sensitivity.** A screening programme that must miss few cases fixes the sensitivity
  it needs, say $0.9$, and accepts whatever specificity results.
- **A cost ratio.** If a false negative is judged $c$ times as costly as a false positive, choose the
  threshold that minimises $c \cdot FN + FP$. The larger $c$ is, the lower the threshold.
- **Youden's index**, $J = \text{sensitivity} + \text{specificity} - 1$, the height of the operating
  point above the chance line of section 5. Maximising $J$ weights the two error rates equally,
  which is rarely the clinical situation but is a common default.
> In every case the threshold is chosen on **validation** data, never on the test set. A threshold
> tuned on the test set makes the test set's metrics optimistic, in the same way that a $\lambda$
> tuned on it would (Meeting 7). The curves below are drawn on the test set only so that they match
> the rest of the lecture.

The choice itself is clinical rather than statistical. The classifier supplies the menu of operating
points; which error is worse, and by how much, is decided by what happens to a woman after each kind
of mistake.

In [ ]:
thresholds = np.round(np.arange(0.05, 0.96, 0.05), 2)
sweep = pd.DataFrame([summarise(y_test, p_test, t) for t in thresholds])
print(sweep[["threshold", "TP", "FN", "FP", "TN", "sensitivity", "specificity", "precision", "accuracy"]]
      .iloc[1:14].to_string(index=False, float_format=lambda v: f"{v:.3f}"))

fine = np.linspace(0.02, 0.98, 97)
J = sweep["sensitivity"] + sweep["specificity"] - 1
print(f"\nYouden's J is largest at t = {sweep['threshold'][J.idxmax()]:.2f}   "
      f"(sensitivity {sweep['sensitivity'][J.idxmax()]:.2f}, specificity {sweep['specificity'][J.idxmax()]:.2f})")
costs = {}
for c in (1, 3, 5):
    costs[c] = np.array([c * summarise(y_test, p_test, t)["FN"] + summarise(y_test, p_test, t)["FP"] for t in fine])
    print(f"cost-minimising threshold when a false negative costs {c}x a false positive: t = {fine[int(np.argmin(costs[c]))]:.2f}")
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
axes[0].plot(sweep["threshold"], sweep["sensitivity"], "o-", color="#c1272d", label="sensitivity")
axes[0].plot(sweep["threshold"], sweep["specificity"], "o-", color="#2b6cb0", label="specificity")
axes[0].plot(sweep["threshold"], sweep["precision"],   "o-", color="#2f8f5b", label="precision")
axes[0].set_xlabel("threshold $t$"); axes[0].set_title("Operating characteristics against the threshold"); axes[0].legend()
for c, col in [(1, "#6e6878"), (3, "#2b6cb0"), (5, "#c1272d")]:
    axes[1].plot(fine, costs[c], lw=2, color=col, label=f"a false negative costs {c}x a false positive")
    axes[1].plot(fine[int(np.argmin(costs[c]))], costs[c].min(), "o", color=col, ms=9)
axes[1].set_xlabel("threshold $t$"); axes[1].set_ylabel("total cost on the test set")
axes[1].set_title("Cost against the threshold"); axes[1].legend(fontsize=10)
for ax in axes: ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

At the threshold $1/2$ the model finds $70\%$ of the women who will develop diabetes. Lowering the
threshold to $0.3$ raises that to $88\%$ at the cost of calling $26\%$ of the healthy women positive;
at $0.2$ no case is missed and $43\%$ of the healthy women are flagged. If a missed case is judged
five times as costly as a false alarm, the cost is minimised near $t = 0.2$; if the two errors are
weighted equally, near $t = 0.5$. Youden's index selects $t \approx 0.3$. None of these is the right
threshold; each is the right threshold for a particular statement about costs.

---
## 5 — The ROC curve and the AUC

Plotting sensitivity against the false positive rate for every threshold at once gives the
**receiver operating characteristic** (ROC) curve.

> The name comes from radar engineering in the 1940s, where the "receiver" was an operator deciding
> whether a signal was an aircraft. The curve is traced out by moving the threshold from $1$, where
> nothing is called positive (the point $(0, 0)$), to $0$, where everything is (the point $(1, 1)$).

Three properties:

- **Each point is an operating point.** The thresholds of section 4 are particular points on the
  curve.
- **The curve depends only on the ranking of the scores.** Sensitivity and FPR are row quantities,
  so the curve is unchanged if the prevalence changes, and it is unchanged by any monotone
  transformation of the scores. The probabilities need not be calibrated for the curve to be correct.
- **The diagonal is chance.** A score that carries no information about the label, such as a random
  number, has sensitivity equal to its false positive rate at every threshold. Every useful classifier
  lies above the diagonal.
### The area under the curve

The **AUC** summarises the curve as a single number between $0.5$ (chance) and $1$ (some threshold
separates the classes perfectly). It has a direct interpretation:

$$\text{AUC} = P\!\left(\text{score of a randomly chosen positive} > \text{score of a randomly chosen negative}\right)$$

> The AUC is a statement about **ranking**. An AUC of $0.9$ means that if one woman who developed
> diabetes and one who did not are drawn at random, the model gives the first the higher probability
> $90\%$ of the time. The cell below checks this by counting pairs.

What the AUC does not say: which operating point will be used, what the error rates are there, and
whether a probability of $0.7$ means that $70\%$ of such women develop diabetes (calibration,
Meeting 12). Two classifiers with the same AUC can have ROC curves that cross, one better at high
sensitivity and the other at high specificity.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, thr = roc_curve(y_test, p_test)
auc = roc_auc_score(y_test, p_test)

# The pairwise check: the fraction of (positive, negative) pairs in which the positive scores higher.
pos, neg = p_test[y_test == 1], p_test[y_test == 0]
pairs = (pos[:, None] > neg[None, :]).mean() + 0.5 * (pos[:, None] == neg[None, :]).mean()
print(f"AUC from the curve: {auc:.3f}     fraction of positive-negative pairs ranked correctly: {pairs:.3f}")
print(f"({len(pos)} positives x {len(neg)} negatives = {len(pos) * len(neg)} pairs)")
fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.plot(fpr, tpr, lw=2.5, color="#5b3a8e", label=f"logistic regression, AUC = {auc:.3f}")
ax.plot([0, 1], [0, 1], "--", color="#6e6878", lw=1.5, label="chance (AUC = 0.5)")
for t, col in [(0.2, "#2f8f5b"), (0.5, "#1a1a1a"), (0.7, "#c1272d")]:
    s = summarise(y_test, p_test, t)
    ax.plot(s["FPR"], s["sensitivity"], "o", color=col, ms=10,
            label=f"threshold {t}:  sens {s['sensitivity']:.2f}, spec {s['specificity']:.2f}")
ax.set_xlabel("false positive rate  (1 − specificity)"); ax.set_ylabel("true positive rate  (sensitivity)")
ax.set_title("ROC curve on the test set"); ax.set_aspect("equal"); ax.legend(loc="lower right", fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 6 — Precision, recall, and prevalence

The **precision–recall curve** plots precision against recall (sensitivity) for every threshold. It
answers a different question from the ROC curve: not "how do the two error rates trade off" but
"of the women the model calls positive, what fraction are?", which is what a clinician acting on the
model's positives experiences.

> **Average precision** (AP) summarises the curve, approximately as the area under it. Its reference
> level is not $0.5$ but the **prevalence**: a classifier that assigns random scores has a precision
> equal to the prevalence at every recall.

Unlike the ROC curve, the precision–recall curve **depends on prevalence**, because precision is a
column quantity. The dependence is exact, and it is Bayes' rule from Meeting 2. At any operating
point with sensitivity (TPR) and false positive rate (FPR), in a population with prevalence $\pi$,

$$\text{precision} = \frac{\text{TPR} \cdot \pi}{\text{TPR} \cdot \pi + \text{FPR} \cdot (1 - \pi)}$$

So the precision–recall curve at any prevalence can be computed from the ROC curve. The cell below
does this for the test set's own prevalence and for two lower ones. The ROC curve is the same in all
three cases.
This is the result of Meeting 2 again — PPV collapses as prevalence falls while sensitivity and
specificity do not move — now for a learned classifier. Two reporting rules follow:

- an ROC curve or an AUC says nothing about precision; for a rare condition, report the
  precision–recall curve or the AP as well
- state the prevalence of the evaluation set with every precision, F1, or accuracy

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

prec, rec, _ = precision_recall_curve(y_test, p_test)
ap      = average_precision_score(y_test, p_test)
pi_test = y_test.mean()
print(f"test-set prevalence {pi_test:.3f}:  average precision {ap:.3f}   (chance level = prevalence)")

def precision_at_prevalence(tpr, fpr, pi):
    """Bayes' rule: the precision at an ROC operating point, in a population with prevalence pi."""
    with np.errstate(invalid="ignore", divide="ignore"):
        return tpr * pi / (tpr * pi + fpr * (1 - pi))

s = summarise(y_test, p_test, 0.5)
print(f"\nat the threshold-1/2 operating point (sensitivity {s['sensitivity']:.2f}, FPR {s['FPR']:.3f}):")
for pi in [pi_test, 0.10, 0.02]:
    print(f"  prevalence {pi:.2f}:  precision {precision_at_prevalence(s['sensitivity'], s['FPR'], pi):.3f}")
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.5))
axes[0].plot(fpr, tpr, lw=2.5, color="#5b3a8e"); axes[0].plot([0, 1], [0, 1], "--", color="#6e6878")
axes[0].set_xlabel("false positive rate"); axes[0].set_ylabel("sensitivity")
axes[0].set_title(f"ROC: identical at every prevalence (AUC {auc:.3f})")
for pi, col in [(pi_test, "#5b3a8e"), (0.10, "#2b6cb0"), (0.02, "#c1272d")]:
    pr = precision_at_prevalence(tpr, fpr, pi)
    ok = ~np.isnan(pr)
    axes[1].plot(tpr[ok], pr[ok], lw=2.5, color=col, label=f"prevalence {pi:.2f}")
    axes[1].axhline(pi, ls=":", color=col, lw=1)
axes[1].set_xlabel("recall (sensitivity)"); axes[1].set_ylabel("precision"); axes[1].set_ylim(0, 1.02)
axes[1].set_title("Precision–recall: one classifier, three prevalences"); axes[1].legend(title="dotted: chance level")
for ax in axes: ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 7 — Class imbalance

A dataset is **imbalanced** when one class is much rarer than the other. Most biomedical
classification problems are, because the condition of interest is usually the minority. Imbalance
affects evaluation, splitting, and training in different ways.

**Evaluation.** Accuracy is inflated by the majority class (section 3) and precision is bounded by
the prevalence (section 6). The row quantities and the AUC are unaffected. Report sensitivity,
specificity, the AUC, and the precision–recall curve, and state the prevalence.

**Splitting.** A random split of an imbalanced dataset can put most of the rare class on one side;
`stratify=y` prevents this. With very few positives, cross-validation (Meeting 11) is used so that
every positive case is tested on once.
**Training.** Logistic regression fitted to data with prevalence $\pi$ learns an intercept
appropriate to $\pi$. Under the model, changing the prevalence of the training data shifts the
log-odds of every sample by the same constant,
$\log\frac{\pi'}{1-\pi'} - \log\frac{\pi}{1-\pi}$, and leaves the other weights, and therefore the
ranking and the ROC curve, approximately unchanged. Two consequences:

- a model trained where the condition is rare and used at the threshold $1/2$ calls almost no one
  positive, although it ranks the women correctly
- the remedy is to move the threshold (section 4), or equivalently to re-weight the classes during
  training so that each class contributes equally to the loss, which shifts the intercept back

> **Class weights** multiply each sample's term in the loss by a factor, larger for the rare class;
> `class_weight="balanced"` in scikit-learn sets the factors inversely proportional to the class
> sizes. **Resampling** — duplicating rare cases or discarding common ones — has the same effect on a
> linear model: it changes the prevalence the model sees, and so the intercept, not the direction
> of $w$.

The cell below trains on a version of the training set in which the positives have been subsampled
to a prevalence of $10\%$, and evaluates on the unchanged test set.

In [ ]:
rng = np.random.default_rng(0)
pos_idx, neg_idx = np.where(y_train == 1)[0], np.where(y_train == 0)[0]
n_pos = int(round(0.10 * len(neg_idx) / 0.90))                       # positives needed for 10% prevalence
keep  = np.concatenate([neg_idx, rng.choice(pos_idx, n_pos, replace=False)])
pi_train, pi_rare = y_train.mean(), y_train[keep].mean()

rare   = LogisticRegression(penalty=None, max_iter=1000).fit(Z_train[keep], y_train[keep])
p_rare = rare.predict_proba(Z_test)[:, 1]

balanced = LogisticRegression(penalty=None, max_iter=1000, class_weight="balanced").fit(Z_train, y_train)
p_bal    = balanced.predict_proba(Z_test)[:, 1]

print(f"training prevalence: original {pi_train:.3f}, subsampled {pi_rare:.3f}\n")
print(f"{'model':<34}{'intercept':>10}{'AUC':>7}{'sens@0.5':>10}{'spec@0.5':>10}{'prec@0.5':>10}")
for name, m, p in [("original training set", model, p_test),
                   ("positives subsampled to 10%", rare, p_rare),
                   ("class_weight='balanced'", balanced, p_bal)]:
    s = summarise(y_test, p, 0.5)
    print(f"{name:<34}{m.intercept_[0]:>10.2f}{roc_auc_score(y_test, p):>7.3f}"
          f"{s['sensitivity']:>10.2f}{s['specificity']:>10.2f}{s['precision']:>10.2f}")

predicted_shift = np.log(pi_rare / (1 - pi_rare)) - np.log(pi_train / (1 - pi_train))
print(f"\nintercept shift predicted from the change in prevalence: {predicted_shift:+.2f}   "
      f"observed: {rare.intercept_[0] - model.intercept_[0]:+.2f}")
print("the other weights, original :", np.round(model.coef_[0], 2))
print("the other weights, subsampled:", np.round(rare.coef_[0], 2))

The subsampled model's AUC is essentially unchanged — it ranks the test women as well as the original
— but at the threshold $1/2$ its sensitivity collapses from $0.70$ to about $0.2$. The intercept moved
by $-1.6$, close to the $-1.56$ predicted from the prevalence ratio, and the threshold did not move
with it; the remaining weights changed by amounts consistent with fitting on 42 rather than 199
positive cases, and the AUC fell only from $0.896$ to $0.882$. Class weighting shifts the intercept
the other way and raises the sensitivity to $0.78$ at the cost of specificity, again with the AUC
unchanged. Neither change alters what the model knows about the women; both alter where the cut
falls.

---
## 8 — Connection to the course

**Meeting 2, completed.** The truth table has become the confusion matrix, the mammogram has become a
model's score, and the three lessons of that meeting have returned unchanged: sensitivity and
specificity belong to the classifier, precision belongs to the classifier *in a population*, and the
threshold is a choice that trades one error for the other. The ROC curve is the picture of that
trade, and Bayes' rule converts it into precision at any prevalence.

**What is still missing.** Every number today was computed once, on 145 women, 50 of them positive.
A sensitivity of $0.70$ estimated from 50 cases has a standard error of about $0.065$; the AUC and
the AP have comparable uncertainty. Meeting 20 attaches confidence intervals to these quantities and
compares two classifiers properly. Whether the probabilities themselves can be trusted — calibration
— is the subject of Meeting 12, and choosing the threshold and the regularisation without touching
the test set is the subject of Meetings 11 and 12.
## What to take away

- The confusion matrix is the truth table of Meeting 2 with a model's decision in place of the test
- Sensitivity, specificity, and their combinations are properties of the classifier; precision, NPV,
  and accuracy also depend on the prevalence of the evaluation set
- Accuracy must be compared with $1 - \text{prevalence}$, the accuracy of predicting the majority
  class
- The threshold sets the operating point; it is chosen on validation data, by a required
  sensitivity, a cost ratio, or a default such as Youden's index
- The ROC curve shows every operating point; the AUC is the probability that a random positive
  outranks a random negative, and ignores prevalence and calibration
- The precision–recall curve depends on prevalence through Bayes' rule, and is the curve to report
  when the positive class is rare
- Imbalance in the training data moves the intercept, not the ranking; move the threshold or
  re-weight the classes
### Wednesday, October 14 — Studio

An end-to-end clinical tabular classifier: the full diabetes dataset, including the two columns set
aside in Meeting 8, in a pipeline that splits, imputes, standardises, fits, chooses a threshold on
validation data, and reports every quantity of today on a test set used once. The Lab 1 defense
quiz takes place at the start of the meeting.

---
## Recommended resources

Everything here is optional and freely available unless marked **(CCNY)**. Each item covers material
from this lecture in a different style or voice; none of it is assigned.

### Start here

| Resource | Why this one | Time |
|---|---|---|
| **[Machine Learning Fundamentals: The Confusion Matrix](https://www.youtube.com/watch?v=Kdsp6soqA7o)** — StatQuest | Section 2 in seven minutes. | 7 min |
| **[Machine Learning Fundamentals: Sensitivity and Specificity](https://www.youtube.com/watch?v=vP06aMoz4v8)** — StatQuest | The row quantities of section 3, and how to choose between classifiers with them. | 12 min |
| **[ROC and AUC, Clearly Explained!](https://www.youtube.com/watch?v=4jRBRDbJemM)** — StatQuest | The construction of section 5, one threshold at a time. | 16 min |
| **[Classification: ROC and AUC](https://developers.google.com/machine-learning/crash-course/classification/roc-and-auc)** — Google Machine Learning Crash Course | A short written treatment with an interactive threshold; the surrounding pages cover accuracy, precision, and recall. | 20 min |
### The literature

| Resource | Why this one | Time |
|---|---|---|
| **[Diagnostic tests 3: receiver operating characteristic plots](https://pmc.ncbi.nlm.nih.gov/articles/PMC2540706/)** — Altman and Bland, *BMJ*, 1994 | One page, the third of the series begun in the Meeting 2 resources: the ROC curve for clinicians. | 10 min |
| **[The Precision-Recall Plot Is More Informative than the ROC Plot When Evaluating Binary Classifiers on Imbalanced Datasets](https://doi.org/10.1371/journal.pone.0118432)** — Saito and Rehmsmeier, *PLoS ONE*, 2015 | The argument of section 6, with simulations across prevalences. Open access. | 45 min |
| **[An introduction to ROC analysis](https://doi.org/10.1016/j.patrec.2005.10.010)** — Fawcett, *Pattern Recognition Letters*, 2006 **(CCNY)** | The standard tutorial: construction, AUC, averaging curves, and the effect of class skew. | 1.5 h |
| **[Metrics and scoring](https://scikit-learn.org/stable/modules/model_evaluation.html)** — scikit-learn User Guide, section 3.4 | The reference for every function used today, with the definitions written out. | reference |